<a href="https://colab.research.google.com/github/03sarath/gcp-ai-agents/blob/main/ai_agents_for_engineers_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Copyright 2025 Psitron Technologies Pvt Ltd

# AI Agents for Engineers (Evolution of AI Agents)

## Overview

This notebook demonstrates 3 different approaches to generating essays using the [Gemini Developer API](https://ai.google.dev/gemini-api/docs) or [Gemini API in Vertex AI](https://cloud.google.com/vertex-ai/generative-ai/docs/overview). Each method illustrates a distinct paradigm for running AI Agents in differing levels of complexity.

1. Step-by-Step Approach With LangChain

## Get started

### Install Gemini SDK and other required packages


In [ ]:
%pip install --upgrade --quiet \
    google-genai \
    langchain \
    langchain-google-genai \
    langchain-google-vertexai \
    langchain-community \
    tavily-python \
    pydantic

### Restart runtime

To use the newly installed packages in this Jupyter runtime, you must restart the runtime. You can do this by running the cell below, which restarts the current kernel.

The restart might take a minute or longer. After it's restarted, continue to the next step.

In [ ]:
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

<div class="alert alert-block alert-warning">
<b>⚠️ The kernel is going to restart. Wait until it's finished before continuing to the next step. ⚠️</b>
</div>


### Configure Tavily

Get an API key for [Tavily](https://tavily.com/), a web search API for Generative AI models.

In [ ]:

import os

os.environ["TAVILY_API_KEY"] = "your-key"

In [ ]:
# # If your API Keys are in Colab Secrets
# import sys

# if "google.colab" in sys.modules:
#     from google.colab import userdata

#     os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

### Configure Gemini Developer API

Get API keys from [Google AI Studio](https://ai.google.dev/gemini-api/docs/api-key) and [Tavily](https://tavily.com/).

In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = "your-key"

In [ ]:
# # If your API Keys are in Colab Secrets
# import sys

# if "google.colab" in sys.modules:
#     from google.colab import userdata

#     os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

### Configure Vertex AI

**Use a Google Cloud Project:** This requires enabling the Vertex AI API in your Google Cloud project.

[Enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com)

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

In [ ]:
PROJECT_ID = "aiagenttest-483711"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "us-central1")

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"

### Import libraries

In [ ]:
from IPython.display import Markdown, display

### Create Client

In [ ]:
from google import genai

client = genai.Client()

Verify which API you are using.

In [ ]:
if not client.vertexai:
    print(f"Using Gemini Developer API.")
elif client._api_client.project:
    print(
        f"Using Vertex AI with project: {client._api_client.project} in location: {client._api_client.location}"
    )
elif client._api_client.api_key:
    print(
        f"Using Vertex AI in express mode with API key: {client._api_client.api_key[:5]}...{client._api_client.api_key[-5:]}"
    )

### Load model

In [ ]:
MODEL_ID = "gemini-2.5-flash"

## Generating Essays Using a Step-by-Step Approach With LangChain

This step demonstrates how to build an essay-writing pipeline using [LangChain](https://www.langchain.com/), the [Gemini API in Google AI Studio](https://ai.google.dev/gemini-api/docs), and [Tavily](https://tavily.com/) for search.

By combining these tools, we create a seamless workflow that plans an essay outline, performs web searches for relevant information, and generates a complete essay draft based on the collected data.

This solution showcases the power of chaining LLM models and external tools to tackle complex tasks with minimal human intervention, providing a robust approach to automated content generation.

<img src="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/workshops/ai-agents/2-langchain-essay.png?raw=1" width="550px">


### Import libraries

In [ ]:
from IPython.display import Markdown, display

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

from langchain_community.tools import TavilySearchResults
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_vertexai import ChatVertexAI

### Initialize Gemini model & search tool

In [ ]:
# ✅ INITIALIZE TAVILY CLIENT
from tavily import TavilyClient
tavily = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

# --- Model setup ---
if client.vertexai:
    model = ChatVertexAI(project=PROJECT_ID, model=MODEL_ID, temperature=0)
else:
    model = ChatGoogleGenerativeAI(model=MODEL_ID, temperature=0)

tavily_tool = TavilySearchResults(max_results=5)

### Define prompt templates and Runnables

In [ ]:
def research_fn(topic):
    results = tavily_tool.invoke({"query": topic})

    # Debug: print to see the actual structure
    print("Research results type:", type(results))
    print("Research results:", results)

    # Handle different return formats
    if isinstance(results, str):
        return results
    elif isinstance(results, list):
        # Check if list contains dicts or strings
        if results and isinstance(results[0], dict):
            return "\n".join([f"- {r.get('content', r.get('snippet', str(r)))}" for r in results])
        else:
            return "\n".join([f"- {r}" for r in results])
    else:
        return str(results)

# --- Prompts ---
outline_prompt = ChatPromptTemplate.from_template(
    """Create a detailed marketing content outline for {topic} that includes:
1. SEO-optimized title
2. Target audience
3. Headers
4. Key points
5. CTA
6. Distribution
7. Social snippets
8. Linking opportunities"""
)

writing_prompt = ChatPromptTemplate.from_template(
    """Based on this outline and research, write a 3-paragraph essay on '{topic}':

Outline:
{outline}

Research:
{research}

Essay:
"""
)

# Final writer chain
writing_chain = writing_prompt | model | StrOutputParser()

### Define the Runnable Chain using [LangChain Expression Language (LCEL)](https://python.langchain.com/docs/how_to/#langchain-expression-language-lcel)

In [ ]:
# Fixed pipeline using RunnablePassthrough.assign()
pipeline = (
    # Start with {"topic": ...}
    RunnablePassthrough.assign(
        # Add "outline" key by running the outline generator
        outline=outline_prompt | model | StrOutputParser()
    )
    # Now we have {"topic": ..., "outline": ...}
    | RunnablePassthrough.assign(
        # Add "research" key
        research=lambda x: research_fn(x["topic"])
    )
    # Now we have {"topic": ..., "outline": ..., "research": ...}
    | writing_chain
)

### Generate the essay

In [ ]:
# Run it
prompt = "Write a 3-paragraph essay about how to implement AI in digital marketing strategies"
result = pipeline.invoke({"topic": prompt})

display(Markdown(result))